# PyTorch Dual Flow Notebook

## Embedded code

In [ ]:
## ssl_disentangling.py
import math
import random
from dataclasses import dataclass
from typing import Callable, Generator, List, Sequence, Tuple, Union

import numpy as np
import torch
from torch import nn


class CheckerBoardMask:
    def __init__(self, axes: List[int], shape: List[int]):
        self._axes_ = list(axes)
        self._shape_ = list(shape)
        mask = np.zeros(shape, dtype=np.float64)
        dimension_count = int(np.prod(shape))
        current_indices = [0] * len(shape)
        mask[tuple(current_indices)] = np.sum(current_indices) % 2
        for _ in range(dimension_count):
            for s in range(len(shape) - 1, -1, -1):
                if current_indices[s] == shape[s] - 1:
                    current_indices[s] = 0
                else:
                    current_indices[s] += 1
                    break
            mask[tuple(current_indices)] = np.sum(current_indices) % 2
        self._mask_ = torch.tensor(mask.reshape(-1), dtype=torch.get_default_dtype())

    def call(self, inputs: torch.Tensor, is_positive: bool = True) -> torch.Tensor:
        mask = self._mask_ if is_positive else 1.0 - self._mask_
        x = inputs
        if x.dim() == 1:
            x = x.unsqueeze(0)
        masked = x * mask.to(device=x.device, dtype=x.dtype)
        return masked


class FlowLayer(nn.Module):
    def __init__(self, shape: List[int], axes: List[int]):
        super().__init__()
        self._shape_ = list(shape)
        self._axes_ = list(axes)

    def compute_jacobian_determinant(self, x: torch.Tensor) -> torch.Tensor:
        return torch.zeros(x.shape[0], device=x.device, dtype=x.dtype)


def _make_dense_stack(num_dims: int, hidden_units: int) -> nn.Sequential:
    stack = nn.Sequential(
        nn.Linear(num_dims, hidden_units),
        nn.ReLU(),
        nn.Linear(hidden_units, num_dims),
    ).to(dtype=torch.get_default_dtype())

    for module in stack.modules():
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    return stack


class ActivationNormalization(FlowLayer):
    def __init__(self, shape: List[int], axes: List[int]):
        super().__init__(shape=shape, axes=axes)
        self._location_ = nn.Parameter(torch.zeros(*shape, dtype=torch.get_default_dtype()))
        self._scale_ = nn.Parameter(torch.ones(*shape, dtype=torch.get_default_dtype()))
        self._scale_constraint_eps = 1e-6

    def _scale_positive(self) -> torch.Tensor:
        return torch.clamp(self._scale_, min=self._scale_constraint_eps)

    def forward(self, inputs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        location = self._location_.to(device=inputs.device, dtype=inputs.dtype)
        scale = self._scale_positive().to(device=inputs.device, dtype=inputs.dtype)
        y_hat = (inputs - location) / scale
        jacobian_determinant = self.compute_jacobian_determinant(inputs)
        return y_hat, jacobian_determinant

    def invert(self, outputs: torch.Tensor) -> torch.Tensor:
        location = self._location_.to(device=outputs.device, dtype=outputs.dtype)
        scale = self._scale_positive().to(device=outputs.device, dtype=outputs.dtype)
        return outputs * scale + location

    def compute_jacobian_determinant(self, x: torch.Tensor) -> torch.Tensor:
        scale = self._scale_positive().to(device=x.device, dtype=x.dtype)
        dimension_count = 1
        for axis in range(1, len(x.shape)):
            if axis not in self._axes_:
                dimension_count *= x.shape[axis]
        jacobian_determinant = -dimension_count * torch.sum(torch.log(scale))
        return torch.zeros(x.shape[0], device=x.device, dtype=x.dtype) + jacobian_determinant


class Reflection(FlowLayer):
    def __init__(self, shape: List[int], axes: List[int], reflection_count: int):
        assert reflection_count >= 1
        super().__init__(shape=shape, axes=axes)
        self._reflection_count_ = reflection_count
        dim = int(np.prod(shape))
        normals = -1.0 + 2.0 * torch.rand(reflection_count, dim, dtype=torch.get_default_dtype())
        normals = normals / normals.norm(dim=1, keepdim=True).clamp_min(1e-6)
        self._reflection_normals_ = nn.Parameter(normals)
        self._inverse_mode_ = False

    def _reflect_(self, x: torch.Tensor) -> torch.Tensor:
        x_new = x
        indices = list(range(self._reflection_count_))
        if self._inverse_mode_:
            indices.reverse()
        for r in indices:
            v_r = self._reflection_normals_[r]
            v_r = v_r / v_r.norm(p=2).clamp_min(1e-6)
            dot = torch.sum(x_new * v_r, dim=-1, keepdim=True)
            x_new = x_new - 2.0 * dot * v_r
        return x_new

    def forward(self, inputs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        old_shape = list(inputs.shape)
        x = inputs.reshape(inputs.shape[0], -1)
        y_hat = self._reflect_(x)
        y_hat = y_hat.reshape(old_shape)
        jacobian_determinant = self.compute_jacobian_determinant(inputs)
        return y_hat, jacobian_determinant

    def invert(self, outputs: torch.Tensor) -> torch.Tensor:
        previous = self._inverse_mode_
        self._inverse_mode_ = True
        reconstructed, _ = self.forward(outputs)
        self._inverse_mode_ = previous
        return reconstructed

    def compute_jacobian_determinant(self, x: torch.Tensor) -> torch.Tensor:
        return torch.zeros(x.shape[0], device=x.device, dtype=x.dtype)


class _PermutationBase(FlowLayer):
    def __init__(self, shape: List[int], axes: List[int], permutation: Sequence[int]):
        super().__init__(shape=shape, axes=axes)
        self._permutation_ = list(permutation)
        self._inverse_permutation_ = list(np.argsort(self._permutation_))

    def forward(self, inputs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        y_hat = inputs[:, self._permutation_]
        jacobian_determinant = self.compute_jacobian_determinant(inputs)
        return y_hat, jacobian_determinant

    def invert(self, outputs: torch.Tensor) -> torch.Tensor:
        return outputs[:, self._inverse_permutation_]

    def compute_jacobian_determinant(self, x: torch.Tensor) -> torch.Tensor:
        return torch.zeros(x.shape[0], device=x.device, dtype=x.dtype)


class CheckerBoardPermutation(_PermutationBase):
    def __init__(self, shape: List[int], axes: List[int]):
        dimension_count = int(np.prod(shape))
        tensor = np.reshape(np.arange(dimension_count), shape)
        rope_values = [None] * dimension_count

        def is_end_of_axis(index: int, limit: int, direction: int) -> bool:
            if direction == 1:
                return index == limit - 1
            return index == 0

        current_indices = [0] * len(shape)
        directions = [1] * len(shape)
        rope_values[0] = tensor[tuple(current_indices)]
        for d in range(dimension_count - 1):
            for s in range(len(shape) - 1, -1, -1):
                if is_end_of_axis(current_indices[s], shape[s], directions[s]):
                    directions[s] = -directions[s]
                else:
                    current_indices[s] += directions[s]
                    break
            rope_values[d + 1] = tensor[tuple(current_indices)]

        for d in range(0, dimension_count - 1, 2):
            rope_values[d], rope_values[d + 1] = rope_values[d + 1], rope_values[d]

        current_indices = [0] * len(shape)
        directions = [1] * len(shape)
        tensor[tuple(current_indices)] = rope_values[0]
        for d in range(dimension_count - 1):
            for s in range(len(shape) - 1, -1, -1):
                if is_end_of_axis(current_indices[s], shape[s], directions[s]):
                    directions[s] = -directions[s]
                else:
                    current_indices[s] += directions[s]
                    break
            tensor[tuple(current_indices)] = rope_values[d + 1]

        permutation = list(np.reshape(tensor, [-1]))
        super().__init__(shape=shape, axes=axes, permutation=permutation)


class Coupling(FlowLayer):
    def __init__(self, shape: List[int], axes: List[int], compute_coupling_parameters: nn.Module, mask: CheckerBoardMask):
        super().__init__(shape=shape, axes=axes)
        self._compute_coupling_parameters_ = compute_coupling_parameters
        self._mask_ = mask

    def _couple_(self, inputs: torch.Tensor, coupling_parameters: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def _decouple_(self, outputs: torch.Tensor, coupling_parameters: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def forward(self, inputs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x_1 = self._mask_.call(inputs=inputs, is_positive=True)
        coupling_parameters = self._compute_coupling_parameters_(x_1)
        y_hat_1 = x_1
        y_hat_2 = self._mask_.call(inputs=self._couple_(inputs=inputs, coupling_parameters=coupling_parameters), is_positive=False)
        y_hat = y_hat_1 + y_hat_2
        jacobian_determinant = self.compute_jacobian_determinant(inputs)
        return y_hat, jacobian_determinant

    def invert(self, outputs: torch.Tensor) -> torch.Tensor:
        y_hat_1 = self._mask_.call(inputs=outputs, is_positive=True)
        coupling_parameters = self._compute_coupling_parameters_(y_hat_1)
        x_1 = y_hat_1
        x_2 = self._mask_.call(inputs=self._decouple_(outputs=outputs, coupling_parameters=coupling_parameters), is_positive=False)
        return x_1 + x_2

    def compute_jacobian_determinant(self, x: torch.Tensor) -> torch.Tensor:
        return torch.zeros(x.shape[0], device=x.device, dtype=x.dtype)


class AdditiveCoupling(Coupling):
    def _couple_(self, inputs: torch.Tensor, coupling_parameters: torch.Tensor) -> torch.Tensor:
        return inputs + coupling_parameters

    def _decouple_(self, outputs: torch.Tensor, coupling_parameters: torch.Tensor) -> torch.Tensor:
        return outputs - coupling_parameters


class FlowModel(nn.Module):
    def __init__(self, flow_layers: List[FlowLayer]):
        super().__init__()
        self.flow_layers = nn.ModuleList(flow_layers)

    def build(self, input_shape=None):
        return self

    def summary(self):
        print("FlowModel(")
        for i, layer in enumerate(self.flow_layers):
            print(f"  ({i}): {layer.__class__.__name__}")
        print(")")

    def forward(self, inputs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        z = inputs
        jacobians = []
        for layer in self.flow_layers:
            z, jacobian = layer(z)
            jacobians.append(jacobian)
        total_j = torch.stack(jacobians, dim=0).sum(dim=0) if jacobians else torch.zeros(inputs.shape[0], device=inputs.device, dtype=inputs.dtype)
        return z, total_j

    def invert(self, outputs: torch.Tensor) -> torch.Tensor:
        x = outputs
        for layer in reversed(self.flow_layers):
            x = layer.invert(x)
        return x


class SupervisedFactorLoss(nn.Module):
    def __init__(self, dimensions_per_factor: List[int]):
        super().__init__()
        factor_count = len(dimensions_per_factor)
        factor_masks = np.zeros((factor_count, int(np.sum(dimensions_per_factor))), dtype=np.float64)
        total = 0
        for idx, dimension_count in enumerate(dimensions_per_factor):
            factor_masks[idx, total : total + dimension_count] = 1.0
            total += dimension_count
        self.__factor_masks__ = torch.tensor(factor_masks, dtype=torch.get_default_dtype())
        self.__dimensions_per_factor__ = list(dimensions_per_factor)

    def forward(self, y_true: torch.Tensor, y_pred: torch.Tensor) -> torch.Tensor:
        d = int(np.sum(self.__dimensions_per_factor__))
        z_tilde_a = y_pred[:, :d]
        z_tilde_b = y_pred[:, d : 2 * d]
        j_a = y_pred[:, 2 * d]
        j_b = y_pred[:, 2 * d + 1]

        y_true_N = torch.matmul(y_true, self.__factor_masks__.to(device=y_true.device, dtype=y_true.dtype))
        eps = 1e-6
        y_true_N = torch.clamp(y_true_N, -1 + eps, 1 - eps)

        term_1 = 0.5 * torch.sum(z_tilde_a.pow(2), dim=1)
        var = 1.0 - y_true_N.pow(2)
        diff = z_tilde_b - y_true_N * z_tilde_a
        term_2 = 0.5 * torch.sum(diff.pow(2) / var + torch.log(var), dim=1)
        loss = term_1 + term_2 - (j_a + j_b)
        return loss.mean()


def reset_random_number_generators(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)


def add_noise(x_1, x_2, noise_standard_deviation):
    return (
        x_1 + np.random.normal(scale=noise_standard_deviation[0], size=x_1.shape),
        x_2 + np.random.normal(scale=noise_standard_deviation[1], size=x_2.shape),
    )


def create_data_set(S: np.ndarray, manifold_function: Callable, noise_standard_deviation: Tuple[float, float]) -> Tuple[np.ndarray, np.ndarray]:
    data_function = lambda x: add_noise(*manifold_function(x), noise_standard_deviation=noise_standard_deviation)
    Y = np.concatenate([np.random.standard_normal(size=[len(S), 1]), (S[:, np.newaxis] - np.mean(S)) / np.std(S)], axis=1)
    Z_1, Z_2 = data_function(x=S)
    Z = np.concatenate([Z_1[:, np.newaxis], Z_2[:, np.newaxis]], axis=1)
    return Z.astype(np.float64), Y.astype(np.float64)


def self_supervised_dual_generator(z1_data, z2_data, batch_size):
    num_samples = len(z1_data)
    y_corr_target = np.tile([0.0, 0.9], (batch_size, 1))
    while True:
        idx = np.random.choice(num_samples, batch_size, replace=False)
        Z1_batch = z1_data[idx]
        Z2_batch = z2_data[idx]
        yield (Z1_batch, Z2_batch), y_corr_target


def construct_layers(stage_count: int, num_dims: int, hidden_units: int = 128) -> List[FlowLayer]:
    layers: List[FlowLayer] = [None] * (6 * stage_count + 1)
    layers[0] = ActivationNormalization(shape=[num_dims], axes=[1])
    for i in range(stage_count):
        layers[6 * i + 1] = Reflection(shape=[num_dims], axes=[1], reflection_count=1)
        mask_1 = CheckerBoardMask(axes=[1], shape=[num_dims])
        compute_coupling_parameters_1 = _make_dense_stack(num_dims=num_dims, hidden_units=hidden_units)
        layers[6 * i + 2] = AdditiveCoupling(shape=[num_dims], axes=[1], compute_coupling_parameters=compute_coupling_parameters_1, mask=mask_1)
        layers[6 * i + 3] = CheckerBoardPermutation(shape=[num_dims], axes=[1])
        compute_coupling_parameters_2 = _make_dense_stack(num_dims=num_dims, hidden_units=hidden_units)
        mask_2 = CheckerBoardMask(axes=[1], shape=[num_dims])
        layers[6 * i + 4] = AdditiveCoupling(shape=[num_dims], axes=[1], compute_coupling_parameters=compute_coupling_parameters_2, mask=mask_2)
        layers[6 * i + 5] = CheckerBoardPermutation(shape=[num_dims], axes=[1])
        layers[6 * i + 6] = ActivationNormalization(shape=[num_dims], axes=[1])
    return layers


def set_global_seed(seed: int):
    reset_random_number_generators(seed=seed)


def sample_curve_data(param_values: np.ndarray, curve_fn: Callable, noise_std: Tuple[float, float]) -> Tuple[np.ndarray, np.ndarray]:
    return create_data_set(S=param_values, manifold_function=curve_fn, noise_standard_deviation=noise_std)


def paired_batch_generator(data_a: np.ndarray, data_b: np.ndarray, batch_size: int):
    return self_supervised_dual_generator(z1_data=data_a, z2_data=data_b, batch_size=batch_size)


def build_flow_layers(stage_count: int, num_dims: int, hidden_units: int = 128) -> List[FlowLayer]:
    return construct_layers(stage_count=stage_count, num_dims=num_dims, hidden_units=hidden_units)


In [ ]:
## plot_lib.py
"""
From the NYU DL course. TODO Check how to reference/cite it later.
"""

import matplotlib as mpl
import numpy as np
import torch
from IPython.display import clear_output
from matplotlib import pyplot as plt


def set_default(figsize=(10, 10), dpi=100):
    plt.style.use(['dark_background', 'bmh'])
    plt.rc('axes', facecolor='k')
    plt.rc('figure', facecolor='k')
    plt.rc('figure', figsize=figsize, dpi=dpi)


def plot_data(X, y, d=0, auto=False, zoom=1, title='Training data (x, y)', point_colors=None):
    X = X.cpu()
    y = y.cpu()
    # If per-point colors provided, use them (accept RGB Nx3 or scalar iterable),
    # otherwise fall back to class-based colormap
    if point_colors is not None:
        try:
            s = plt.scatter(X.numpy()[:, 0], X.numpy()[:, 1], c=point_colors, s=20)
        except Exception:
            s = plt.scatter(X.numpy()[:, 0], X.numpy()[:, 1], c=point_colors, s=20, cmap=plt.cm.Spectral)
    else:
        s = plt.scatter(X.numpy()[:, 0], X.numpy()[:, 1], c=y, s=20, cmap=plt.cm.Spectral)
    plt.axis('square')
    plt.axis(np.array((-1.1, 1.1, -1.1, 1.1)) * zoom)
    if auto is True: plt.axis('equal')
    plt.axis('off')

    _m, _c = 0, '.35'
    plt.axvline(0, ymin=_m, color=_c, lw=1)
    plt.axhline(0, xmin=_m, color=_c, lw=1)
    plt.title(title)
    return s


def plot_model(X, y, model, point_colors=None):
    model.cpu()
    mesh = torch.arange(-1.1, 1.11, 0.01)
    xx, yy = torch.meshgrid(mesh, mesh, indexing='xy')
    with torch.no_grad():
        data = torch.stack((xx.reshape(-1), yy.reshape(-1)), dim=1)
        Z = model(data)
    Z = Z.argmax(dim=1).reshape(xx.shape)
    plt.contourf(xx.numpy(), yy.numpy(), Z, cmap=plt.cm.Spectral, alpha=0.3)
    plot_data(X, y, point_colors=point_colors)
    plt.title('Model decision boundaries')


def plot_embeddings(X, y, model, zoom=10, point_colors=None):
    import torch
    # Use forward hook to get internal embeddings of the second-last layer
    layer_outputs = {}

    def get_layer_outputs(name):
        def hook(module, input, output):
            layer_outputs[name] = output.detach()
        return hook

    # guard: expect second-last layer to be a linear with 2 outputs
    layer = model[-2]
    if isinstance(layer, torch.nn.Linear) and layer.out_features == 2:
        handle = layer.register_forward_hook(get_layer_outputs("low_dim_embeddings"))
        try:
            with torch.no_grad():
                model(X)  # populate hook
            emb = layer_outputs["low_dim_embeddings"].cpu().numpy()

            # If point_colors provided, use those (Nx3 array in [0,1]), else color by class
            if point_colors is not None:
                # ensure numpy array with shape (N,3)
                pc = point_colors
            else:
                pc = None

            # First draw decision boundary contour in the low-dim embedding space
            last_layer = model[-1]
            mesh = torch.arange(-1.1, 1.1, 0.01) * zoom
            xx, yy = torch.meshgrid(mesh, mesh, indexing='ij')
            with torch.no_grad():
                grid_data = torch.stack((xx.reshape(-1), yy.reshape(-1)), dim=1)
                Z = last_layer(grid_data)
            Z = Z.argmax(dim=1).reshape(xx.shape)
            plt.contourf(xx.numpy(), yy.numpy(), Z, cmap=plt.cm.Spectral, alpha=0.25)

            # Then plot points on top with full opacity so stored RGBs remain vivid
            if pc is not None:
                plt.scatter(emb[:, 0], emb[:, 1], c=pc, s=25, edgecolors='none', alpha=1.0, zorder=10)
            else:
                plt.scatter(emb[:, 0], emb[:, 1], c=y.cpu().numpy(), s=25, cmap=plt.cm.Spectral, edgecolors='none', alpha=1.0, zorder=10)

            plt.axis('square')
            plt.axis(np.array((-1.1, 1.1, -1.1, 1.1)) * zoom)
            plt.axis('off')
            plt.title('Low dim embeddings')
        finally:
            handle.remove()
    else:
        print("Cannot plot embeddings: second-last layer is not Linear with out_features==2")


def acc(l, y):
    score, predicted = torch.max(l, 1)
    return (y == predicted).sum().float() / len(y)


def overwrite(string):
    print(string)
    clear_output(wait=True)


def plot_2d_energy_levels(X, y, energy, v=None, l=None, point_colors=None):
    xx, yy, F, k, K = energy
    if not v: vmin = vmax = None
    else: vmin, vmax = v
    if not l: levels = None
    else: levels = torch.arange(l[0], l[1], l[2])
    plt.figure(figsize=(12, 10))
    plt.pcolormesh(xx.numpy(), yy.numpy(), F, vmin=vmin, vmax=vmax)
    plt.colorbar()
    cnt = plt.contour(xx.numpy(), yy.numpy(), F, colors='w', linewidths=1, levels=levels)
    plt.clabel(cnt, inline=True, fontsize=10, colors='w')
    s = plot_data(X, y, point_colors=point_colors)
    plt.legend(*s.legend_elements(), title='Classes', loc='lower right')
    plt.axvline(color='0.55', lw=1)
    plt.axhline(color='0.55', lw=1)
    plt.axis([-1.5, 1.5, -1.5, 1.5])
    ȳ = torch.zeros(K).int(); ȳ[k] = 1
    plt.title(f'Free energy F(x, y = {ȳ.tolist()})')


def plot_3d_energy_levels(X, y, energy, v=None, l=None, cbl=None, point_colors=None):
    xx, yy, F, k, K = energy
    if not v: vmin = vmax = None
    else: vmin, vmax = v
    if not l: levels = None
    else: levels = torch.arange(l[0], l[1], l[2])
    fig = plt.figure(figsize=(9.5, 6), facecolor='k')
    ax = fig.add_subplot(projection='3d')
    cnt = ax.contour(xx.numpy(), yy.numpy(), F, levels=levels, vmin=vmin, vmax=vmax)
    # Use provided per-point colors if available
    if point_colors is not None:
        try:
            ax.scatter(X[:,0], X[:,1], zs=0, c=point_colors)
        except Exception:
            ax.scatter(X[:,0], X[:,1], zs=0, c=y, cmap=plt.cm.Spectral)
    else:
        ax.scatter(X[:,0], X[:,1], zs=0, c=y, cmap=plt.cm.Spectral)
    ax.xaxis.set_pane_color(color=(0,0,0))
    ax.yaxis.set_pane_color(color=(0,0,0))
    ax.zaxis.set_pane_color(color=(0,0,0))

    vmin, vmax = cnt.get_clim()
    ax.set_zlim3d(vmin, vmax)
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    if not cbl: cbl = l
    else: cbl = torch.arange(cbl[0], cbl[1], cbl[2])
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cnt.cmap)
    sm.set_array([])
    fig.colorbar(sm, ticks=cbl, ax=ax)
    ȳ = torch.zeros(K).int(); ȳ[k] = 1
    plt.title(f'Free energy F(x, y = {ȳ.tolist()})')
    plt.tight_layout()
    return fig, ax

## Embedded code instead of imports

In [ ]:
import os
import sys
from pathlib import Path
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import torch
from ipywidgets import interact

sys.path.insert(0, str(Path.cwd().parent))
# from res.plot_lib import *
# from pysrc.ssl_disentangling import (
#     ActivationNormalization,
#     AdditiveCoupling,
#     CheckerBoardMask,
#     CheckerBoardPermutation,
#     FlowModel,
#     Reflection,
#     SupervisedFactorLoss,
#     build_flow_layers,
#     paired_batch_generator,
#     sample_curve_data,
#     set_global_seed,
# )

torch.set_default_dtype(torch.float64)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
save_plots = True
output_dir = os.path.join("output_04-spiral_classification", "pytorch_dual_disentangle")
os.makedirs(output_dir, exist_ok=True)
set_default()

## Create Base Spiral Data

In [ ]:
seed = 12345
torch.manual_seed(seed)
np.random.seed(seed)
N = 4096  # num_samples_per_class
n = 2     # input dimensions
K = 1     # num_classes
d = 100   # num_hidden_units

In [ ]:
t = torch.linspace(0, 1, N)
amplitude = 0.8 * t + 0.2
X = list()
y = list()
for k in range(K):
    theta = (2 * t + k) * 2 * np.pi / K + 0.2 * torch.randn(N)
    X.append(torch.stack((amplitude * theta.sin(), amplitude * theta.cos()), dim=1))
    y.append(torch.zeros(N, dtype=torch.long).fill_(k))
X = torch.cat(X)
y = torch.cat(y)

print("Shapes:")
print("X:", tuple(X.size()))
print("y:", tuple(y.size()))

In [ ]:
from matplotlib import colors as mpl_colors

X_np = X.cpu().numpy()
y_np = y.cpu().numpy()
r = np.sqrt(X_np[:, 0] ** 2 + X_np[:, 1] ** 2)
r_min, r_max = r.min(), r.max()
r_norm = (r - r_min) / (r_max - r_min + 1e-9)
K = int(y.max() + 1) if isinstance(y, torch.Tensor) else int(np.max(y) + 1)
hues = np.arange(K) / float(K)
v_min, v_max = 0.05, 0.95
values = v_min + (v_max - v_min) * r_norm
saturation = 0.9
h = hues[y_np]
s = np.full_like(h, saturation, dtype=float)
v = values
hsv = np.stack([h, s, v], axis=1)
rgb = mpl_colors.hsv_to_rgb(hsv)
point_colors = rgb

In [ ]:
plot_data(X, y, point_colors=point_colors)
if save_plots:
    plt.savefig(os.path.join(output_dir, "data.png"), bbox_inches='tight')

## Prepare Dataset A

In [ ]:
num_samples = 4096
batch_size = 128
num_dims = 2

N = num_samples
param_values = np.linspace(0, 1, N)

def curve_a_fn(u: np.ndarray):
    return (
        (0.8 * u + 0.2) * np.sin(2 * u * 2 * np.pi),
        (0.8 * u + 0.2) * np.cos(2 * u * 2 * np.pi),
    )

noise_std_a = [0.02, 0.02]

data_a, _ = sample_curve_data(
    param_values=param_values,
    curve_fn=curve_a_fn,
    noise_std=noise_std_a,
)

print("Dataset A structure:")
print(f"data_a shape: {data_a.shape}")
print(f"param_values shape: {param_values.shape}")
print("curve_a_fn defined: yes")

## Prepare Dataset B

In [ ]:
data_b = np.copy(data_a)

def curve_b_fn(u: np.ndarray):
    x = u * 2.0 - 1.0
    y = x**3 - 0.5 * x - 0.5
    return (x, y)

noise_std_b = [0.02, 0.02]

data_b, _ = sample_curve_data(
    param_values=param_values,
    curve_fn=curve_b_fn,
    noise_std=noise_std_b,
)

batch_size = 64
pair_gen = paired_batch_generator(data_a=data_a, data_b=data_b, batch_size=batch_size)
(batch_a, batch_b), corr_target = next(pair_gen)

print("batch_a shape:", batch_a.shape)
print("batch_b shape:", batch_b.shape)

In [ ]:
plt.scatter(param_values, np.zeros_like(param_values), color='blue', alpha=0.5, label='param_values')
plt.title('Curve Parameter')

In [ ]:
np.array(curve_a_fn(param_values)).shape

In [ ]:
curve_a = curve_a_fn(param_values)
plt.scatter(curve_a[0], curve_a[1], color='blue', alpha=0.5, label='curve_a')
plt.title('Curve A')

In [ ]:
np.array(curve_b_fn(param_values)).shape

In [ ]:
curve_b = curve_b_fn(param_values)
plt.scatter(curve_b[0], curve_b[1], color='blue', alpha=0.5, label='curve_b')
plt.title('Curve B')

In [ ]:
plt.scatter(data_a[:,0], data_a[:,1], color='blue', alpha=0.5, label='data_a')
plt.title('Dataset A')

In [ ]:
plt.scatter(data_b[:,0], data_b[:,1], color='blue', alpha=0.5, label='data_b')
plt.title('Dataset B')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].scatter(batch_a[:, 0], batch_a[:, 1], color='blue', alpha=0.5)
axs[0].set_title('Batch A')
axs[0].set_xlabel('Dim 1')
axs[0].set_ylabel('Dim 2')
axs[0].axis('equal')
axs[1].scatter(batch_b[:, 0], batch_b[:, 1], color='green', alpha=0.5)
axs[1].set_title('Batch B')
axs[1].set_xlabel('Dim 1')
axs[1].set_ylabel('Dim 2')
axs[1].axis('equal')
plt.tight_layout()
plt.show()

## Dual Flow Training

In [ ]:
set_global_seed(seed=123)
layers_a = build_flow_layers(stage_count=10, num_dims=num_dims)
model_a = FlowModel(layers_a).to(device)
model_a.build(input_shape=[num_samples, num_dims])

layers_b = build_flow_layers(stage_count=2, num_dims=num_dims)
model_b = FlowModel(layers_b).to(device)
model_b.build(input_shape=[num_samples, num_dims])

print("Model A and Model B successfully built.")
model_a.summary()

In [ ]:
class DualPairModel(torch.nn.Module):
    def __init__(self, model_a, model_b):
        super().__init__()
        self.model_a = model_a
        self.model_b = model_b

    def forward(self, input_a, input_b):
        output_a, jac_a = self.model_a(input_a)
        output_b, jac_b = self.model_b(input_b)
        return torch.cat([output_a, output_b, jac_a.unsqueeze(1), jac_b.unsqueeze(1)], dim=1)

class DiagonalPredictor(torch.nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * self.weight

def build_predictor(dim=1, hidden_dim=64):
    return DiagonalPredictor(dim)

predictor_a2b = build_predictor(dim=num_dims).to(device)
predictor_b2a = build_predictor(dim=num_dims).to(device)
dual_model = DualPairModel(model_a, model_b)

import torch.nn.functional as F
class EBMJEPALoss(torch.nn.Module):
    def __init__(self, predictor_a2b, predictor_b2a, lambda_jac=1.0, lambda_prior=0.5, lambda_sparse=0.1):
        super().__init__()
        self.predictor_a2b = predictor_a2b
        self.predictor_b2a = predictor_b2a
        self.lambda_jac = lambda_jac
        self.lambda_prior = lambda_prior
        self.lambda_sparse = lambda_sparse

    def forward(self, dual_model_outputs):
        d = (dual_model_outputs.shape[1] - 2) // 2
        z_a, z_b = dual_model_outputs[:, :d], dual_model_outputs[:, d:2*d]
        jac_a, jac_b = dual_model_outputs[:, 2*d], dual_model_outputs[:, 2*d+1]

        # Use L1 loss for the prediction error to encourage sparsity
        loss_a2b = F.l1_loss(self.predictor_a2b(z_a), z_b, reduction='none').sum(dim=-1)
        loss_b2a = F.l1_loss(self.predictor_b2a(z_b), z_a, reduction='none').sum(dim=-1)

        # Laplace Prior (L1 on latents) to force axis-alignment
        prior_loss = self.lambda_prior * (torch.sum(torch.abs(z_a), dim=-1) + torch.sum(torch.abs(z_b), dim=-1))
        
        # Sparsity penalty on the predictor weights
        sparse_loss = self.lambda_sparse * (torch.sum(torch.abs(self.predictor_a2b.weight)) + torch.sum(torch.abs(self.predictor_b2a.weight)))
        
        return (loss_a2b + loss_b2a - self.lambda_jac * (jac_a + jac_b) + prior_loss).mean() + sparse_loss

loss_function = EBMJEPALoss(predictor_a2b, predictor_b2a, lambda_jac=1.0, lambda_prior=0.5, lambda_sparse=0.1)
optimizer = torch.optim.Adam(list(model_a.parameters()) + list(model_b.parameters()) + \
                             list(predictor_a2b.parameters()) + list(predictor_b2a.parameters()), lr=0.001)

data_a_all = torch.tensor(data_a, device=device, dtype=torch.float64)
data_b_all = torch.tensor(data_b, device=device, dtype=torch.float64)

history = []
steps_per_epoch = 50
epochs = 500

In [ ]:
for epoch in range(epochs):
    epoch_loss = 0.0
    for _ in range(steps_per_epoch):
        (batch_a_np, batch_b_np), corr_target_np = next(pair_gen)
        batch_a_t = torch.tensor(batch_a_np, device=device, dtype=torch.float64)
        batch_b_t = torch.tensor(batch_b_np, device=device, dtype=torch.float64)

        predictions = dual_model(batch_a_t, batch_b_t)
        loss = loss_function(predictions)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    history.append(epoch_loss / steps_per_epoch)
    if epoch % 50 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:03d} | loss={history[-1]:.6f}")

print("Training completed")

In [ ]:
continue_training = True
if continue_training:
    # Set models back to training mode since they were set to eval() for inference
    dual_model.train()
    predictor_a2b.train()
    predictor_b2a.train()

    additional_epochs = 50
    start_epoch = len(history)
    print(f"Resuming training from epoch {start_epoch} for {additional_epochs} more epochs...")

    for epoch in range(start_epoch, start_epoch + additional_epochs):
        epoch_loss = 0.0
        for _ in range(steps_per_epoch):
            (batch_a_np, batch_b_np), corr_target_np = next(pair_gen)
            batch_a_t = torch.tensor(batch_a_np, device=device, dtype=torch.float64)
            batch_b_t = torch.tensor(batch_b_np, device=device, dtype=torch.float64)

            predictions = dual_model(batch_a_t, batch_b_t)
            loss = loss_function(predictions)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        history.append(epoch_loss / steps_per_epoch)
        
        # Print progress every 10 epochs
        if (epoch - start_epoch) % 10 == 0 or epoch == start_epoch + additional_epochs - 1:
            print(f"Epoch {epoch:03d} | loss={history[-1]:.6f}")

    print("Continued training completed")


In [ ]:
checkpoint_path = os.path.join(output_dir, "dual_model_checkpoint_2.pt")

torch.save(
    {
        "epoch": epoch,
        "model_a_state_dict": model_a.state_dict(),
        "model_b_state_dict": model_b.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "batch_size": batch_size,
        "seed": seed,
    },
    checkpoint_path,
 )

print(f"Checkpoint saved to: {checkpoint_path}")

In [ ]:
checkpoint_path = os.path.join(output_dir, "dual_model_checkpoint_2.pt")

checkpoint = torch.load(checkpoint_path, map_location=device)

model_a.load_state_dict(checkpoint["model_a_state_dict"])
model_b.load_state_dict(checkpoint["model_b_state_dict"])

if "optimizer_state_dict" in checkpoint:
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

history = checkpoint.get("history", history)
batch_size = checkpoint.get("batch_size", batch_size)
seed = checkpoint.get("seed", seed)

dual_model.to(device).eval()

print(f"Loaded checkpoint from: {checkpoint_path}")
print(f"history_len={len(history)}, batch_size={batch_size}, seed={seed}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history)
plt.title('Training loss')
plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

# Four-Panel Inference Check

In [ ]:
data_a_np = data_a if isinstance(data_a, np.ndarray) else data_a.numpy()
data_b_np = data_b if isinstance(data_b, np.ndarray) else data_b.numpy()

with torch.no_grad():
    output_a, _ = model_a(torch.tensor(data_a_np, device=device, dtype=torch.float64))
    output_b, _ = model_b(torch.tensor(data_b_np, device=device, dtype=torch.float64))

output_a = output_a.detach().cpu().numpy()
output_b = output_b.detach().cpu().numpy()
color_code = (param_values - np.min(param_values)) / (np.max(param_values) - np.min(param_values) + 1e-12)

def dual_geometry_reshaping_view(dim_1: float = None, dim_2: float = None):
    fig, axs = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle('Self-Supervised Dual Geometry Reshaping')

    has_marker = (dim_1 is not None) and (dim_2 is not None)
    if has_marker:
        output_marker = torch.tensor([[dim_1, dim_2]], device=device, dtype=torch.float64)
        with torch.no_grad():
            input_a_marker = model_a.invert(output_marker).detach().cpu().numpy()[0]
            input_b_marker = model_b.invert(output_marker).detach().cpu().numpy()[0]

    axs[0].scatter(data_a_np[:, 0], data_a_np[:, 1], c=color_code, cmap='turbo', s=10, alpha=0.85)
    if has_marker:
        axs[0].scatter([input_a_marker[0]], [input_a_marker[1]], color='white', s=100, edgecolors='white', linewidths=1.0, zorder=5)
    axs[0].set_title('Input Space A')
    axs[0].set_xlabel('Dim 1')
    axs[0].set_ylabel('Dim 2')
    axs[0].axis('equal')

    axs[1].scatter(output_a[:, 0], output_a[:, 1], c=color_code, cmap='turbo', s=10, alpha=0.85)
    if has_marker:
        axs[1].scatter([dim_1], [dim_2], color='white', s=100, edgecolors='white', linewidths=1.0, zorder=5)
    axs[1].set_title('Output Space A')
    axs[1].set_xlabel('Dim 1')
    axs[1].set_ylabel('Dim 2')
    axs[1].axis('equal')

    axs[2].scatter(output_b[:, 0], output_b[:, 1], c=color_code, cmap='turbo', s=10, alpha=0.85)
    if has_marker:
        axs[2].scatter([dim_1], [dim_2], color='white', s=100, edgecolors='white', linewidths=1.0, zorder=5)
    axs[2].set_title('Output Space B')
    axs[2].set_xlabel('Dim 1')
    axs[2].set_ylabel('Dim 2')
    axs[2].axis('equal')

    axs[3].scatter(data_b_np[:, 0], data_b_np[:, 1], c=color_code, cmap='turbo', s=10, alpha=0.85)
    if has_marker:
        axs[3].scatter([input_b_marker[0]], [input_b_marker[1]], color='white', s=100, edgecolors='white', linewidths=1.0, zorder=5)
    axs[3].set_title('Input Space B')
    axs[3].set_xlabel('Dim 1')
    axs[3].set_ylabel('Dim 2')
    axs[3].axis('equal')

    plt.tight_layout()
    plt.show()

# interact(
#     dual_geometry_reshaping_view,
#     dim_1=(-2.0, 2.0, 0.1),
#     dim_2=(-2.0, 2.0, 0.1),
# )
dual_geometry_reshaping_view()

In [ ]:
# Determine Matching and Unique Dimensions
w_a2b = predictor_a2b.weight.detach().cpu().numpy()
w_b2a = predictor_b2a.weight.detach().cpu().numpy()

threshold = 0.5
matching_dims = [i for i, w in enumerate(w_a2b) if abs(w) > threshold]
unique_dims = [i for i, w in enumerate(w_a2b) if abs(w) <= threshold]

print("Predictor A -> B weights:", np.round(w_a2b, 3))
print("Predictor B -> A weights:", np.round(w_b2a, 3))
print(f"\\nMatching (Shared) Dimensions ({len(matching_dims)}): {matching_dims}")
print(f"Space-Unique Dimensions ({len(unique_dims)}): {unique_dims}")
